# YOLO12s + L-Ghost + EMA-32 — frozen proposed-model training (Multi-seed)

Notebook Kaggle ini mengikuti gaya penulisan dan susunan cell `yolo12-ghost-ema-v4 (2).ipynb`.
Arsitektur dan semantic pretrained transfer berasal dari branch `feat/yolo12s-lghost-ema32`,
dibekukan pada commit `14cbeb1b9a136c17da4ff0acfe01a3f625cd5e3e`.

Aktifkan GPU dan Internet di Kaggle. Runner meng-clone repository, checkout commit tersebut, dan memverifikasi
semantic transfer dari `yolo12s.pt` untuk target RDD2022 lima kelas sebelum epoch pertama.
`LGhostDetectionTrainer` memakai loss/optimizer deteksi standar dengan penambahan jadwal pruning LG dan sinkronisasi mask.
LG memakai G=4, RA=0.6; pruning setelah 32, 64, dan 96 epoch. Ini adaptasi L-GhostNet, bukan replikasi utuh paper.

Hyperparameter mengikuti notebook acuan: **160 epoch, imgsz 640, batch 16, nbs 64, SGD, lr0 0.01,
momentum 0.937, weight_decay 0.0005, patience 0, workers 2, seed 42/123/456, cos_lr=False**.
Alur tetap: clone/install → pre-flight → training → val → test (atau prediction jika tanpa label) → ZIP.
EMA-32 attention ada di P3/P4.
ModelEMA rata-rata bobot Ultralytics tetap terpisah dari EMA attention. Best checkpoint dapat berasal dari sebelum pruning selesai;
tahap pruning dicatat dalam laporan evaluasi. File ini belum dilatih dan output lama telah dibersihkan.

Multi-seed: seed 42, 123, 456 dengan hyperparameter identik. Hasil per seed disimpan sebagai ZIP terpisah, lalu diringkas sebagai mean ± std.

In [ ]:
# 1. Clone snapshot arsitektur yang dibekukan dan install repository lokal.
import json
import platform
import subprocess
import sys
from pathlib import Path

if 'ultralytics' in sys.modules:
    raise RuntimeError('Restart session Kaggle sebelum install untuk menghindari modul dari repo lain.')

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
SOURCE_BRANCH = 'feat/yolo12s-lghost-ema32'
FROZEN_COMMIT = '14cbeb1b9a136c17da4ff0acfe01a3f625cd5e3e'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022-lghost-ema32'

def log_section(title: str) -> None:
    print()
    print('=' * 88)
    print(title)
    print('=' * 88)

def run(command, cwd=None) -> None:
    print('>>', subprocess.list2cmdline([str(item) for item in command]))
    subprocess.run(command, cwd=cwd, check=True)

log_section('CLONE FROZEN PROPOSED ARCHITECTURE')
if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', SOURCE_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    origin = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=REPO_DIR, text=True).strip()
    status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
    assert origin == REPO_URL, 'Folder clone berisi repo berbeda.'
    assert not status, 'Ada perubahan lokal; gunakan folder clone baru.'
    run(['git', 'fetch', 'origin', SOURCE_BRANCH], cwd=REPO_DIR)
run(['git', 'checkout', '--detach', FROZEN_COMMIT], cwd=REPO_DIR)

REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert REPO_COMMIT == FROZEN_COMMIT, f'Unexpected source revision: {REPO_COMMIT}'
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPO_DIR)])
sys.path.insert(0, str(REPO_DIR))

REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(
    f'repository={REPO_URL}\nbranch={SOURCE_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8'
)

import torch
import ultralytics

assert Path(ultralytics.__file__).resolve().is_relative_to(REPO_DIR.resolve()), 'Import bukan dari repo clone.'
assert torch.cuda.is_available(), 'Aktifkan GPU accelerator di Kaggle sebelum training.'
DEVICE = 0
log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Source branch: {SOURCE_BRANCH}')
print(f'Commit      : {REPO_COMMIT}')
print(f'GPU         : {torch.cuda.get_device_name(0)}')

# Protokol run seed 42 lama: PyTorch 2.10.0+cu128 dan Tesla T4. Perbedaan tidak menghentikan run,
# tetapi wajib dicatat di log sesi (README_SOP.md).
if torch.__version__ != '2.10.0+cu128' or not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    print('WARNING: environment berbeda dari run seed 42 lama — catat di log sesi README_SOP.md.')


CLONE FROZEN PROPOSED ARCHITECTURE
>> git clone --branch feat/yolo12s-lghost-ema32 https://github.com/danial2015/yolo-aceh-rdd2022.git /kaggle/working/yolo-aceh-rdd2022-lghost-ema32


Cloning into '/kaggle/working/yolo-aceh-rdd2022-lghost-ema32'...


>> git checkout --detach 14cbeb1b9a136c17da4ff0acfe01a3f625cd5e3e
>> /usr/bin/python3 -m pip install --quiet -e /kaggle/working/yolo-aceh-rdd2022-lghost-ema32


HEAD is now at 14cbeb1b feat: add YOLO12s L-Ghost-only ablation config


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 3.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

ENVIRONMENT
Python      : 3.13.15
PyTorch     : 2.11.0+cu128
Ultralytics : 8.4.127
Source branch: feat/yolo12s-lghost-ema32
Commit      : 14cbeb1b9a136c17da4ff0acfe01a3f625cd5e3e
GPU         : Tesla T4


In [ ]:
# 2. Dataset dan hyperparameter. Pertahankan protokol yang sama dengan baseline.
import zipfile

from ultralytics.utils import YAML

DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix')
DATA_YAML = WORKDIR / 'ch_rdd2022_frozen.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12s-lghost-ema32.yaml'
RUNS_DIR = WORKDIR / 'runs'
EXPERIMENT_NAME = 'yolo12s_lghost_ema32_semantic_pretrained_ch_rdd2022'

EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS = 0, 2
# Multi-seed: hyperparameter identik untuk setiap seed; hanya SEED yang berbeda.
SEEDS = [42, 123, 456]
# Batas sesi Kaggle 12 jam. Bila tiga seed tidak muat dalam satu sesi, jalankan sebagian,
# misalnya RUN_SEEDS = [42] lalu [123, 456] pada sesi berikutnya.
# Seed 42 memakai hasil run lama (lihat README_SOP.md); sesi ini menjalankan:
RUN_SEEDS = [123, 456]
# Opsional: folder hasil sesi sebelumnya (Kaggle input) untuk agregasi mean ± std.
# '/kaggle/input' dicari rekursif: hasil seed 42 lama dan output sesi sebelumnya otomatis ikut.
EXTRA_RESULT_DIRS = ['/kaggle/input']

assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists(), f'Model YAML tidak ditemukan: {MODEL_YAML}'
for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / split / 'images'
    assert image_dir.exists(), f'Image directory tidak ditemukan: {image_dir}'
    image_count = sum(
        path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'} for path in image_dir.rglob('*')
    )
    assert image_count > 0, f'Tidak ada image pada split {split}'
    print(f'{split:5s}: {image_count:,} images')

DATA_YAML.write_text(
    f'''path: {DATA_ROOT.as_posix()}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''',
    encoding='utf-8',
)

semantic_spec = YAML.load(MODEL_YAML)['semantic_pretrained']
assert semantic_spec['source_layers'] == 22
assert semantic_spec['source_detect_from'] == [14, 17, 20]
assert semantic_spec['target_detect_from'] == [15, 19, 22]

EXPECTED_TRANSFER = {
    'mode': 'semantic',
    'source_tensors': 699,
    'target_tensors': 767,
    'semantic_candidates': 679,
    'exact_transferred_tensors': 673,
    'destination_missing_tensors': 8,
    'shape_mismatch_tensors': 6,
    'new_module_tensors': 88,
    'uninitialized_target_tensors': 94,
    'exact_transferred_parameter_tensors': 340,
    'exact_transferred_parameter_elements': 8515088,
}
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RUN_MANIFEST = WORKDIR / f'{EXPERIMENT_NAME}_planned_run.json'
RUN_MANIFEST.write_text(
    json.dumps(
        {
            'repository': REPO_URL,
            'source_branch': SOURCE_BRANCH,
            'frozen_commit': REPO_COMMIT,
            'model_yaml': str(MODEL_YAML),
            'dataset_yaml': str(DATA_YAML),
            'epochs': EPOCHS,
            'imgsz': IMGSZ,
            'batch': BATCH,
            'nbs': NBS,
            'optimizer': OPTIMIZER,
            'lr0': LR0,
            'momentum': MOMENTUM,
            'weight_decay': WEIGHT_DECAY,
            'patience': PATIENCE,
            'seeds': SEEDS,
            'run_seeds': RUN_SEEDS,
            'expected_pretrained_transfer': EXPECTED_TRANSFER,
        },
        indent=2,
    ),
    encoding='utf-8',
)
print(f'Dataset YAML: {DATA_YAML}')
print(json.dumps(EXPECTED_TRANSFER, indent=2))

# Pemeriksaan awal (sebelum training berjam-jam): seed yang sudah punya hasil vs seed sesi ini.
PREVIOUS_SEEDS = sorted({
    json.loads(p.read_text(encoding='utf-8'))['seed']
    for d in EXTRA_RESULT_DIRS if Path(d).is_dir()
    for p in Path(d).rglob(f'{EXPERIMENT_NAME}_seed*_evaluation_metrics.json')
})
print(f'Seed dengan hasil di input : {PREVIOUS_SEEDS}')
print(f'Seed dijalankan sesi ini   : {RUN_SEEDS}')
if 42 not in PREVIOUS_SEEDS and 42 not in RUN_SEEDS:
    raise FileNotFoundError('Hasil seed 42 lama tidak ditemukan. Add Input dataset seed42-previous (README_SOP.md langkah A).')
if set(PREVIOUS_SEEDS) & set(RUN_SEEDS):
    raise ValueError(f'Seed {sorted(set(PREVIOUS_SEEDS) & set(RUN_SEEDS))} sudah punya hasil; '
                     'hapus dari RUN_SEEDS agar kuota GPU tidak terbuang.')
pending = sorted(set(SEEDS) - set(PREVIOUS_SEEDS) - set(RUN_SEEDS))
if pending:
    print(f'Catatan: seed {pending} dijadwalkan untuk sesi berikutnya.')

train: 3,502 images
val  : 438 images
test : 438 images
Dataset YAML: /kaggle/working/ch_rdd2022_frozen.yaml
{
  "mode": "semantic",
  "source_tensors": 699,
  "target_tensors": 767,
  "semantic_candidates": 679,
  "exact_transferred_tensors": 673,
  "destination_missing_tensors": 8,
  "shape_mismatch_tensors": 6,
  "new_module_tensors": 88,
  "uninitialized_target_tensors": 94,
  "exact_transferred_parameter_tensors": 340,
  "exact_transferred_parameter_elements": 8515088
}
Seed dengan hasil di input : [42]
Seed dijalankan sesi ini   : [123, 456]


In [ ]:
# 3. Validasi graph proposed lima kelas sebelum trainer L-Ghost dijalankan.
from ultralytics import YOLO
from ultralytics.nn.modules import EMA, LGhostBottleneck
from ultralytics.nn.modules.lghost import LearnedGroupConv
from ultralytics.models.yolo.detect.lghost import LGhostDetectionTrainer
from ultralytics.nn.tasks import DetectionModel

log_section('PROPOSED MODEL PRE-FLIGHT')
inspection_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=False)
ema_layers = [layer.i for layer in inspection_model.model if isinstance(layer, EMA)]
lghost_layers = [layer.i for layer in inspection_model.model if isinstance(layer, LGhostBottleneck)]
assert ema_layers == [15, 19], f'Unexpected EMA layers: {ema_layers}'
assert lghost_layers == [16, 20], f'Unexpected LGhostBottleneck layers: {lghost_layers}'
assert inspection_model.model[-1].nc == 5
assert inspection_model.model[-1].f == [15, 19, 22]
print(f'EMA layers       : {ema_layers}')
print(f'LGhostBottleneck layers : {lghost_layers}')
print(f'Detect inputs    : {inspection_model.model[-1].f}')
inspection_model.info(detailed=False, verbose=True, imgsz=IMGSZ)
del inspection_model
torch.cuda.empty_cache()
# Model YOLO dibuat ulang per seed di cell berikut.


PROPOSED MODEL PRE-FLIGHT
Overriding model.yaml nc=80 with nc=5
EMA layers       : [15, 19]
LGhostBottleneck layers : [16, 20]
Detect inputs    : [15, 19, 22]
YOLO12s-lghost-ema32 summary: 314 layers, 8,738,223 parameters, 8,738,207 gradients, 23.3 GFLOPs


In [ ]:
# 4. Training dengan pruning L-Ghost per seed: train → validation → test → ZIP.
TRAINING_TRANSFER_REPORT = {}
TRANSFER_REPORT_PATH = None

def verify_semantic_pretrained_transfer(trainer) -> None:
    global TRANSFER_REPORT_PATH
    report = dict(getattr(trainer.model, 'pretrained_transfer_report', {}))
    differences = {
        key: {'expected': expected, 'actual': report.get(key)}
        for key, expected in EXPECTED_TRANSFER.items()
        if report.get(key) != expected
    }
    if differences:
        raise AssertionError(f'Semantic pretrained audit failed: {json.dumps(differences, indent=2)}')
    TRAINING_TRANSFER_REPORT.clear()
    TRAINING_TRANSFER_REPORT.update(report)
    TRANSFER_REPORT_PATH = Path(trainer.save_dir) / 'semantic_pretrained_transfer.json'
    TRANSFER_REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
    log_section('SEMANTIC PRETRAINED TRANSFER VERIFIED')
    print(json.dumps(report, indent=2))

import gc, shutil, statistics

def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def seed_paths(seed):
    name = f'{EXPERIMENT_NAME}_seed{seed}'
    return (name, WORKDIR / f'{name}_evaluation_metrics.json',
            WORKDIR / f'{name}_config.json', WORKDIR / f'{name}_results.zip')

def write_seed_zip(zip_path, archive_items):
    tmp_path = zip_path.with_suffix('.zip.tmp')
    files = []
    for item in archive_items:
        if item is None or not item.exists():
            continue
        files.extend([item] if item.is_file() else [p for p in item.rglob('*') if p.is_file()])
    files = list(dict.fromkeys(files))
    with zipfile.ZipFile(tmp_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for file_path in files:
            archive.write(file_path, file_path.relative_to(WORKDIR).as_posix())
    tmp_path.replace(zip_path)  # ZIP final = penanda seed selesai.
    return len(files)

def prepare_seed(seed):
    name, evaluation_json, _, zip_path = seed_paths(seed)
    if zip_path.is_file() and evaluation_json.is_file():
        log_section(f'SEED {seed} SUDAH SELESAI — DILEWATI')
        return json.loads(evaluation_json.read_text(encoding='utf-8'))
    # Hapus sisa run seed ini yang terputus agar artefak seed tidak tercampur.
    for suffix in ('', '_val', '_test', '_test_predictions'):
        shutil.rmtree(RUNS_DIR / f'{name}{suffix}', ignore_errors=True)
    return None

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

def run_seed(SEED):
    name, EVALUATION_JSON, FINAL_MANIFEST, ZIP_PATH = seed_paths(SEED)
    done = prepare_seed(SEED)
    if done is not None:
        return done

    # Jangan memanggil model.load() di sini. Trainer membangun target nc=5
    # dan memanggil BaseModel.load() sekali dengan pretrained='yolo12s.pt'.
    TRAINING_TRANSFER_REPORT.clear()
    model = YOLO(str(MODEL_YAML), task='detect')
    model.add_callback('on_pretrain_routine_end', verify_semantic_pretrained_transfer)
    log_section(f'TRAINING STARTED — FROZEN YOLO12S + L-GHOST + EMA-32 — SEED {SEED}')
    print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
    model.train(
        trainer=LGhostDetectionTrainer,
        compile=False,
        time=None,
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        nbs=NBS,
        device=DEVICE,
        workers=WORKERS,
        project=str(RUNS_DIR),
        name=name,
        exist_ok=False,
        pretrained='yolo12s.pt',
        optimizer=OPTIMIZER,
        lr0=LR0,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        cos_lr=False,
        patience=PATIENCE,
        seed=SEED,
        plots=True,
        verbose=True,
    )
    assert TRAINING_TRANSFER_REPORT, 'Transfer report tidak pernah diterima oleh callback trainer.'
    transfer_report = dict(TRAINING_TRANSFER_REPORT)
    transfer_report_path = TRANSFER_REPORT_PATH
    RUN_DIR = Path(model.trainer.save_dir)
    BEST_PT = Path(model.trainer.best)
    LAST_PT = Path(model.trainer.last)
    assert BEST_PT.exists() and LAST_PT.exists(), 'Checkpoint training tidak lengkap.'
    print(f'Run directory: {RUN_DIR}')
    print(f'Best weights : {BEST_PT}')
    del model
    free_gpu_memory()

    # Evaluasi best.pt.
    log_section(f'BEST CHECKPOINT EVALUATION — SEED {SEED}')
    best_model = YOLO(str(BEST_PT))
    pruning_stages = [int(m.stage) for m in best_model.model.modules() if isinstance(m, LearnedGroupConv)]
    print(f'LG pruning stages best.pt: {pruning_stages} (fully pruned = 3)')

    val_metrics = best_model.val(
        data=str(DATA_YAML),
        split='val',
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=f'{name}_val',
        exist_ok=True,
        plots=True,
    )
    VAL_DIR = Path(val_metrics.save_dir)
    EVALUATION_REPORT = {'seed': SEED, 'validation': metric_summary(val_metrics), 'lg_pruning_stages': pruning_stages}
    del val_metrics

    test_label_dir = DATA_ROOT / 'test' / 'labels'
    if test_label_dir.exists() and any(test_label_dir.rglob('*.txt')):
        test_metrics = best_model.val(
            data=str(DATA_YAML),
            split='test',
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            project=str(RUNS_DIR),
            name=f'{name}_test',
            exist_ok=True,
            plots=True,
        )
        TEST_DIR = Path(test_metrics.save_dir)
        EVALUATION_REPORT['test'] = metric_summary(test_metrics)
        del test_metrics
    else:
        TEST_DIR = RUNS_DIR / f'{name}_test_predictions'
        best_model.predict(
            source=str(DATA_ROOT / 'test' / 'images'),
            imgsz=IMGSZ,
            device=DEVICE,
            conf=0.25,
            save=True,
            save_txt=True,
            project=str(RUNS_DIR),
            name=TEST_DIR.name,
            exist_ok=True,
        )
        EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_DIR)}
    del best_model
    free_gpu_memory()

    # Simpan metadata seed ini, lalu buat ZIP artefak Kaggle.
    EVALUATION_REPORT['pretrained_transfer'] = transfer_report
    EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
    FINAL_MANIFEST.write_text(
        json.dumps(
            {
                'repository': REPO_URL,
                'source_branch': SOURCE_BRANCH,
                'frozen_commit': REPO_COMMIT,
                'model_yaml': str(MODEL_YAML),
                'dataset_yaml': str(DATA_YAML),
                'hyperparameters': {
                    'epochs': EPOCHS,
                    'imgsz': IMGSZ,
                    'batch': BATCH,
                    'nbs': NBS,
                    'optimizer': OPTIMIZER,
                    'lr0': LR0,
                    'momentum': MOMENTUM,
                    'weight_decay': WEIGHT_DECAY,
                    'patience': PATIENCE,
                    'seed': SEED,
                    'seeds': SEEDS,
                },
                'pretrained_transfer': transfer_report,
                'best_checkpoint': str(BEST_PT),
                'last_checkpoint': str(LAST_PT),
                'evaluation': EVALUATION_REPORT,
            },
            indent=2,
        ),
        encoding='utf-8',
    )
    file_count = write_seed_zip(ZIP_PATH, [
        RUN_DIR,
        VAL_DIR,
        TEST_DIR,
        DATA_YAML,
        MODEL_YAML,
        REPO_METADATA,
        RUN_MANIFEST,
        FINAL_MANIFEST,
        transfer_report_path,
        EVALUATION_JSON,
    ])
    log_section(f'SEED {SEED} COMPLETED')
    print(json.dumps(EVALUATION_REPORT, indent=2))
    print(f'ZIP created: {ZIP_PATH} ({file_count} files)')
    return EVALUATION_REPORT

In [ ]:
# 5. Jalankan semua seed secara berurutan; memori GPU dibersihkan setelah setiap seed.
ALL_REPORTS = {}
for seed in RUN_SEEDS:
    ALL_REPORTS[seed] = run_seed(seed)
    free_gpu_memory()
    if torch.cuda.is_available():
        print(f'GPU memory setelah seed {seed}: {torch.cuda.memory_allocated() / 1024**2:.0f} MiB allocated')


TRAINING STARTED — FROZEN YOLO12S + L-GHOST + EMA-32 — SEED 123
epochs=160, imgsz=640, batch=16, nbs=64, optimizer=SGD, lr0=0.01, seed=123
New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ch_rdd2022_frozen.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=160, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015,

In [ ]:
# 6. Agregasi mean ± std seluruh seed yang sudah selesai, lalu ZIP ringkasan.
pattern = f'{EXPERIMENT_NAME}_seed*_evaluation_metrics.json'
found = {}
for path in sorted(WORKDIR.glob(pattern)) + [p for d in EXTRA_RESULT_DIRS for p in sorted(Path(d).rglob(pattern))]:
    report = json.loads(path.read_text(encoding='utf-8'))
    found.setdefault(report['seed'], (path, report))

def mean_std(values):
    return {'mean': statistics.mean(values), 'std': statistics.stdev(values) if len(values) > 1 else 0.0,
            'values': values}

METRIC_KEYS = ('precision', 'recall', 'map50', 'map50_95')
SUMMARY = {'experiment': EXPERIMENT_NAME, 'seeds': sorted(found)}
for split_key in ('validation', 'test'):
    # Split test tanpa label (prediction only) tidak memiliki metrik sehingga tidak diagregasi.
    reports = [found[s][1][split_key] for s in sorted(found) if 'map50' in found[s][1].get(split_key, {})]
    if reports:
        SUMMARY[split_key] = {k: mean_std([r[k] for r in reports]) for k in METRIC_KEYS}

SUMMARY_JSON = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.json'
SUMMARY_JSON.write_text(json.dumps(SUMMARY, indent=2), encoding='utf-8')
SUMMARY_CSV = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.csv'
rows = ['seed,split,' + ','.join(METRIC_KEYS)]
for s in sorted(found):
    for split_key in ('validation', 'test'):
        r = found[s][1].get(split_key, {})
        if 'map50' in r:
            rows.append(f'{s},{split_key},' + ','.join(f'{r[k]:.6f}' for k in METRIC_KEYS))
for split_key in ('validation', 'test'):
    if split_key in SUMMARY:
        for stat in ('mean', 'std'):
            rows.append(f'{stat},{split_key},' + ','.join(f'{SUMMARY[split_key][k][stat]:.6f}' for k in METRIC_KEYS))
SUMMARY_CSV.write_text('\n'.join(rows) + '\n', encoding='utf-8')

SUMMARY_ZIP = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.zip'
with zipfile.ZipFile(SUMMARY_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in [SUMMARY_JSON, SUMMARY_CSV, *(found[s][0] for s in sorted(found))]:
        archive.write(path, path.name)

log_section(f'MULTI-SEED SUMMARY — {len(found)}/{len(SEEDS)} SEED')
missing = sorted(set(SEEDS) - set(found))
if missing:
    print(f'WARNING: seed belum selesai: {missing}')
print(SUMMARY_CSV.read_text(encoding='utf-8'))
from IPython.display import FileLink, display
display(FileLink(SUMMARY_ZIP))
for s in sorted(found):
    seed_zip = seed_paths(s)[3]
    if seed_zip.is_file():
        display(FileLink(seed_zip))


MULTI-SEED SUMMARY — 3/3 SEED
seed,split,precision,recall,map50,map50_95
42,validation,0.886738,0.819000,0.886868,0.600332
42,test,0.883231,0.830711,0.872306,0.592890
123,validation,0.854229,0.839321,0.873148,0.602193
123,test,0.862383,0.835296,0.863348,0.583661
456,validation,0.848869,0.831804,0.898077,0.604912
456,test,0.860288,0.818608,0.858645,0.567622
mean,validation,0.863279,0.830041,0.886031,0.602479
std,validation,0.020493,0.010274,0.012485,0.002303
mean,test,0.868634,0.828205,0.864767,0.581391
std,test,0.012685,0.008622,0.006940,0.012786



/kaggle/working/yolo12s_lghost_ema32_semantic_pretrained_ch_rdd2022_multiseed_summary.zip

/kaggle/working/yolo12s_lghost_ema32_semantic_pretrained_ch_rdd2022_seed123_results.zip

/kaggle/working/yolo12s_lghost_ema32_semantic_pretrained_ch_rdd2022_seed456_results.zip